# 01 – Kaggle EDA and Feature Analysis

This notebook explores the `Store Item Demand Forecasting Challenge` dataset and
the synthetic/demo series used when Kaggle data is not available.

It is designed to run in **two modes**:

1. **Kaggle mode** – when `data/raw/train.csv` exists.
2. **Demo mode** – when Kaggle data is missing; falls back to a synthetic series
   generated by the `demandcast.data.synthetic` module or the small demo CSV
   under `assets/demo/demo_series.csv`.


In [1]:
from pathlib import Path

import pandas as pd

import sys
PROJECT_ROOT = Path.cwd().resolve().parents[0]
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from demandcast.data import (
    build_target_series,
    generate_demo_series,
    load_kaggle_train_df,
    locate_kaggle_train_csv,
)
from demandcast.features import (
    FeatureConfig,
    build_time_series_features,
)
from demandcast.evaluation.metrics import rmse, mae, mape, smape, wape
from demandcast.logging_utils import get_logger
from demandcast.paths import DEMO_DATA_DIR

logger = get_logger(__name__)


## Load Kaggle data or fall back to demo series

This cell tries to load the Kaggle competition data. If it is not found, it
falls back to the synthetic demo series (or the committed demo CSV).

In [2]:
try:
    csv_path = locate_kaggle_train_csv()
    raw_df = load_kaggle_train_df(csv_path)
    source = f"Kaggle CSV – {csv_path}"
except FileNotFoundError:
    demo_csv = DEMO_DATA_DIR / "demo_series.csv"
    if demo_csv.exists():
        demo_df = pd.read_csv(demo_csv)
        demo_df["date"] = pd.to_datetime(demo_df["ds"])
        demo_df["store"] = demo_df.get("store", 1)
        demo_df["item"] = demo_df.get("item", 1)
        demo_df["sales"] = demo_df.get("y", demo_df.get("sales", 1.0))
        raw_df = demo_df[["date", "store", "item", "sales"]].copy()
        source = f"Demo CSV – {demo_csv}"
    else:
        raw_df = generate_demo_series()
        raw_df["date"] = raw_df["ds"]
        raw_df["store"] = raw_df.get("store", 1)
        raw_df["item"] = raw_df.get("item", 1)
        raw_df["sales"] = raw_df.get("y", 1.0)
        raw_df = raw_df[["date", "store", "item", "sales"]].copy()
        source = "Synthetic demo series (generated on the fly)"

logger.info("Using data source: %s", source)
raw_df.head()

2026-01-26 21:35:47,716 | INFO | demandcast.data.load_data | Loading Kaggle train CSV from D:\Project\demandcast-inventory-optimizer\data\raw\train.csv
2026-01-26 21:35:48,159 | INFO | demandcast.data.load_data | Loaded training data: 913000 rows, 10 stores, 50 items, date range 2013-01-01 -> 2017-12-31
2026-01-26 21:35:48,159 | INFO | demandcast.data.load_data | Missing sales percentage after cleaning: 0.000%
2026-01-26 21:35:48,161 | INFO | __main__ | Using data source: Kaggle CSV – D:\Project\demandcast-inventory-optimizer\data\raw\train.csv


,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10


## Build a univariate target series

We collapse the store–item panel into a single series for a default
`(store, item)` pair or aggregate as needed.

In [3]:
series_df = build_target_series(raw_df)
series_df.head()

2026-01-26 21:35:48,215 | INFO | demandcast.data.load_data | Built target series in mode=single_sku with 1826 rows (2013-01-01 -> 2017-12-31)


,ds,y,store,item
0,2013-01-01,13,1,1
1,2013-01-02,11,1,1
2,2013-01-03,14,1,1
3,2013-01-04,13,1,1
4,2013-01-05,10,1,1


## Feature engineering

We apply the time-series feature pipeline (calendar features, lags, and rolling
statistics) to construct a supervised learning dataset for downstream models
such as LightGBM.

In [4]:
features_cfg = FeatureConfig()
features_df = build_time_series_features(series_df, config=features_cfg)
features_df.head()

2026-01-26 21:35:48,234 | INFO | demandcast.features.build_features | Built time series features: 1826 rows, 17 columns


,ds,y,store,item,dow,doy,week,month,year,y_lag_1,y_lag_7,y_lag_14,y_lag_28,y_roll_mean_7,y_roll_std_7,y_roll_mean_28,y_roll_std_28
0,2013-01-01,13,1,1,1,1,1,1,2013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2013-01-02,11,1,1,2,2,1,1,2013,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2013-01-03,14,1,1,3,3,1,1,2013,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2013-01-04,13,1,1,4,4,1,1,2013,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2013-01-05,10,1,1,5,5,1,1,2013,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Quick baseline metrics

To get a rough sense of difficulty, we can compute very simple baselines such
as **naïve lag-1** or **moving average** and evaluate them with RMSE, MAE,
MAPE, sMAPE, and WAPE. For brevity, the exact code is omitted here; the
core implementation lives in `src/demandcast/evaluation/metrics.py` and the
scripts in `scripts/` provide repeatable CLI entry points.